# Video 10: Molecular Docking

**AI for Drug Discovery series | DigitalSreeni**

---

In the previous videos we scored candidate EGFR inhibitor molecules using ADMET models and generated improved molecules using scaffold decoration and a GPT-based generative model. Our best compounds scored between 7 and 10 out of 10 on ADMET properties.

But ADMET scoring predicts drug-like properties from molecular fingerprints. It does not tell us whether a molecule actually fits into the EGFR binding pocket, in the right orientation, with a favorable binding energy. That is what molecular docking does.

In this notebook we take three compounds from the series and dock them against the EGFR kinase domain crystal structure:

- **CHEMBL382797**: our original lead compound from Video 7b (ADMET score 7/10)
- **Scaffold_dec_best**: the best scaffold-decorated variant from Video 8 (ADMET score 8/10)
- **SelfiesGPT_rank1**: the top-ranked molecule from the SelfiesGPT generative model in Video 8 (ADMET score 9/10)

The docking scores will tell us whether the ADMET improvements from the previous videos translate into better binding at the protein level.

**What this notebook covers:**
- Fetching the EGFR crystal structure from the Protein Data Bank
- Preparing the protein for docking using OpenBabel
- Generating 3D ligand conformers and preparing ligand PDBQT files using RDKit and Meeko
- Defining the binding site search box
- Running AutoDock Vina for all three compounds
- Comparing docking scores across compounds
- Visualizing the best docked poses interactively using py3Dmol

## Cell 1: Installs, Imports, and Configuration

Molecular docking requires several tools that are not in the default Colab environment. We install them here.

**AutoDock Vina** is the docking engine. It takes a prepared protein structure and a prepared ligand and searches for the binding pose with the lowest free energy of binding. We download the precompiled binary directly from the AutoDock Vina GitHub releases page.

**OpenBabel** is a chemical file format converter. We use it to prepare the protein structure by removing water molecules and converting the PDB file to the PDBQT format that Vina requires.

**Meeko** is a Python library for ligand preparation. It converts RDKit molecule objects into PDBQT format, handling atom typing and partial charge assignment that Vina needs to compute binding energies.

**py3Dmol** is a lightweight 3D molecular viewer that renders directly in the notebook. We use it to visualize the docked poses.

A note on exhaustiveness: AutoDock Vina has a parameter called exhaustiveness that controls how thoroughly it searches the binding site. The default is 8. We use 4 in this notebook to keep runtime around 60 seconds per compound, which is appropriate for a tutorial. For publication-quality results you would use 16 or higher.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 1 -- installs, imports, configuration
# Downloads Vina binary, installs OpenBabel, Meeko, py3Dmol, and all Python dependencies

# AutoDock Vina binary -- pip installs Python bindings only, binary must be downloaded separately
!wget -q https://github.com/ccsb-scripps/AutoDock-Vina/releases/download/v1.2.5/vina_1.2.5_linux_x86_64 -O /usr/local/bin/vina
!chmod +x /usr/local/bin/vina

# OpenBabel for protein preparation and format conversion
!apt-get install -qq openbabel

# Python packages
!pip install rdkit vina meeko gemmi py3Dmol -q

import os
import subprocess
import time
import warnings
import requests
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import py3Dmol

from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem, Descriptors, Draw
from meeko import MoleculePreparation, PDBQTWriterLegacy

# Suppress RDKit warnings (noisy stereochemistry and valence warnings not relevant here)
RDLogger.DisableLog('rdApp.*')
warnings.filterwarnings('ignore')

# Results directory on Google Drive -- persists across sessions
RESULTS_DIR = Path('/content/drive/MyDrive/ColabNotebooks/AI_for_drug_discovery/Video10_docking/results')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Font for plots
plt.rcParams['font.family'] = 'DejaVu Sans'

# Docking configuration
PDB_ID        = '1IEP'       # EGFR kinase domain co-crystallized with erlotinib
EXHAUSTIVENESS = 4           # use 16+ for publication quality; 4 is sufficient for tutorial
NUM_MODES      = 5           # number of binding poses to generate per compound

# Binding site box centered on the EGFR ATP binding pocket in 1IEP
# These coordinates are derived from the co-crystallized erlotinib ligand position
BOX_CENTER = (22.0, 53.0, 18.0)
BOX_SIZE   = (20.0, 20.0, 20.0)

# Confirm Vina binary
vina_check = subprocess.run(['vina', '--version'], capture_output=True, text=True)
print(f'Vina:      {vina_check.stdout.strip()}')

obabel_check = subprocess.run(['obabel', '--version'], capture_output=True, text=True)
print(f'OpenBabel: {obabel_check.stdout.split(chr(10))[0]}')
print(f'Results:   {RESULTS_DIR}')
print('Configuration complete.')

## Cell 2: Define Compounds

We define the three compounds we will dock, carrying forward their SMILES strings and ADMET scores from the previous videos.

CHEMBL382797 is our original series lead. It scored 7 out of 10 on ADMET and belongs to the quinazoline scaffold class, the same structural family as erlotinib and gefitinib.

The scaffold decoration compound is the best result from the RDKit fragment decoration exercise in Video 8. It adds a hydroxyl group to the quinazoline core of CHEMBL382797, improving the ADMET score to 8 out of 10.

The SelfiesGPT compound is the top-ranked molecule generated by the GPT model trained on EGFR inhibitors in Video 8. It scored 9 out of 10 on ADMET and has a structurally distinct profile from the quinazoline compounds, featuring a trifluoromethyl chain and a more flexible backbone. The docking result for this compound will be particularly interesting: a high ADMET score does not guarantee good binding, and this is exactly the kind of cross-validation that docking provides.

In [ ]:
# Cell 2 -- define the three compounds carried forward from the series

compounds = [
    {
        'name':        'CHEMBL382797',
        'smiles':      'FC(F)(F)c1ccc(Nc2ncc3cc(Nc4ccccc4)ccc3n2)cc1',
        'admet_score': 7,
        'source':      'Video 7b lead compound',
        'color':       '#1B2A4A',
    },
    {
        'name':        'Scaffold_dec_best',
        'smiles':      'Oc1ccc(Nc2ccc3nc(Nc4ccc(C(F)(F)F)cc4)ncc3c2)cc1',
        'admet_score': 8,
        'source':      'Video 8 scaffold decoration',
        'color':       '#0D9488',
    },
    {
        'name':        'SelfiesGPT_rank1',
        'smiles':      'CNC1=CC=CNC1=CC=CN=CNC=CC=CC(F)(F)F',
        'admet_score': 9,
        'source':      'Video 8 SelfiesGPT generative model',
        'color':       '#F59E0B',
    },
]

df_compounds = pd.DataFrame(compounds)
print('Compounds to dock:\n')
print(df_compounds[['name', 'admet_score', 'source']].to_string(index=False))

# Quick validation: confirm all SMILES parse and embed correctly
print('\nSMILES validation:')
for c in compounds:
    mol = Chem.MolFromSmiles(c['smiles'])
    if mol is None:
        print(f"  {c['name']}: INVALID SMILES")
        continue
    mol_h = Chem.AddHs(mol)
    result = AllChem.EmbedMolecule(mol_h, randomSeed=42)
    status = 'OK' if result != -1 else '3D EMBEDDING FAILED'
    print(f"  {c['name']}: {status}  MW={Descriptors.MolWt(mol):.1f}  atoms={mol.GetNumAtoms()}")

## Cell 3: Fetch and Inspect the EGFR Crystal Structure

We fetch PDB entry 1IEP from the RCSB Protein Data Bank. This structure is the EGFR kinase domain co-crystallized with erlotinib, a first-generation quinazoline EGFR inhibitor. It was resolved at 2.6 angstrom resolution and is one of the most widely used structures for EGFR docking studies.

We chose this structure for two reasons. First, erlotinib belongs to the same quinazoline scaffold class as two of our three compounds, so the binding pocket geometry is appropriate. Second, having a co-crystallized ligand gives us a reference point: we can compare where our compounds dock against where erlotinib sits in the crystal structure.

A raw PDB file contains the protein, the co-crystallized ligand, and water molecules. For docking we want only the protein. We will handle that separation in the next cell during protein preparation.

In [ ]:
# Cell 3 -- fetch the EGFR crystal structure from the Protein Data Bank

pdb_path = Path(f'{PDB_ID}.pdb')

url      = f'https://files.rcsb.org/download/{PDB_ID}.pdb'
response = requests.get(url)
pdb_path.write_text(response.text)

# Count record types to confirm the structure downloaded correctly
lines       = response.text.split('\n')
atom_lines  = [l for l in lines if l.startswith('ATOM')]
hetatm_lines = [l for l in lines if l.startswith('HETATM')]
remark_lines = [l for l in lines if l.startswith('REMARK')]

print(f'PDB entry:               {PDB_ID}')
print(f'File size:               {pdb_path.stat().st_size:,} bytes')
print(f'Protein ATOM records:    {len(atom_lines)}')
print(f'HETATM records (ligand + water): {len(hetatm_lines)}')
print(f'\nThis structure contains the EGFR kinase domain co-crystallized with erlotinib.')
print(f'We will remove the co-crystallized ligand and water molecules during protein preparation.')

## Cell 4: Prepare the Protein

Before docking, the protein structure needs to be prepared. Raw PDB files are not directly usable by AutoDock Vina for several reasons.

First, water molecules need to be removed. Crystal structures contain water molecules that fill the binding site. We want our ligand to displace those waters during docking, not compete with them.

Second, the file needs to be converted to PDBQT format. PDBQT is an extension of PDB format that includes partial atomic charges and atom type information. AutoDock Vina uses these to compute the scoring function that estimates binding energy.

We use OpenBabel for this preparation. The `-xr` flag removes non-polar hydrogens to reduce file size, and the `--partialcharge gasteiger` flag assigns Gasteiger partial charges, which are the standard for AutoDock Vina calculations.

In [ ]:
# Cell 4 -- prepare the protein: remove waters and convert to PDBQT format

protein_pdbqt = Path(f'{PDB_ID}_protein.pdbqt')

result = subprocess.run([
    'obabel', str(pdb_path),
    '-O', str(protein_pdbqt),
    '-xr',
    '--partialcharge', 'gasteiger',
    '-xp',
], capture_output=True, text=True)

if protein_pdbqt.exists():
    print(f'Protein PDBQT prepared: {protein_pdbqt}')
    print(f'File size: {protein_pdbqt.stat().st_size:,} bytes')

    # Count ATOM lines in PDBQT to confirm protein is intact
    pdbqt_atoms = [l for l in protein_pdbqt.read_text().split('\n') if l.startswith('ATOM')]
    print(f'ATOM records in PDBQT: {len(pdbqt_atoms)}')
    print('Protein preparation complete.')
else:
    print('Protein preparation failed.')
    print(result.stderr)

## Cell 5: Prepare the Ligands

Each compound needs to be converted from a 2D SMILES string into a 3D structure in PDBQT format before docking.

This process has three steps. First, we use RDKit to generate a 3D conformer from the SMILES string. This places each atom in three-dimensional space using a distance geometry algorithm and then optimizes the geometry using a molecular mechanics force field.

Second, we add hydrogen atoms. Hydrogen atoms are usually omitted from SMILES strings but they are important for docking because they affect the shape and electrostatics of the molecule.

Third, we use Meeko to convert the RDKit molecule into PDBQT format. Meeko assigns atom types and partial charges that AutoDock Vina uses in its scoring function, and it identifies which bonds in the molecule are rotatable so that Vina can sample different conformations during the docking search.

In [ ]:
# Cell 5 -- prepare all three ligands: 3D conformer generation and PDBQT conversion

def prepare_ligand(name, smiles):
    """Generate 3D conformer and write PDBQT file for a compound. Returns path or None on failure."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        print(f'  {name}: invalid SMILES')
        return None

    # Add hydrogens and generate 3D conformer
    mol = Chem.AddHs(mol)
    result = AllChem.EmbedMolecule(mol, randomSeed=42)
    if result == -1:
        print(f'  {name}: 3D embedding failed')
        return None

    # Optimize geometry with MMFF force field
    AllChem.MMFFOptimizeMolecule(mol)

    # Prepare PDBQT with Meeko
    preparator = MoleculePreparation()
    mol_setups  = preparator.prepare(mol)
    pdbqt_string, is_ok, error_msg = PDBQTWriterLegacy.write_string(mol_setups[0])

    if not is_ok:
        print(f'  {name}: Meeko preparation failed: {error_msg}')
        return None

    pdbqt_path = Path(f'{name}.pdbqt')
    pdbqt_path.write_text(pdbqt_string)
    return pdbqt_path


print('Preparing ligands...\n')
ligand_paths = {}

for c in compounds:
    path = prepare_ligand(c['name'], c['smiles'])
    if path:
        ligand_paths[c['name']] = path
        print(f"  {c['name']}: OK  ({path.stat().st_size} bytes)")

print(f'\n{len(ligand_paths)}/{len(compounds)} ligands prepared successfully.')

## Cell 6: Define the Binding Site Search Box

AutoDock Vina does not search the entire protein surface. We define a rectangular search box centered on the region where we expect the ligand to bind, and Vina searches only within that box. This makes the calculation much faster and more focused.

For 1IEP, the binding site is the ATP binding pocket of the EGFR kinase domain. We know its location because the co-crystallized erlotinib ligand sits there. The box coordinates used here are derived from the center of mass of erlotinib in the crystal structure.

The box is 20 angstroms on each side. This is large enough to cover the full ATP binding pocket and allow the ligand to explore different binding orientations, but small enough to keep the search efficient.

In general, when you do not have a co-crystallized ligand to reference, you would use a binding site prediction tool such as fpocket or SiteMap to identify the most likely binding cavity on the protein surface.

In [ ]:
# Cell 6 -- define and visualize the binding site search box

cx, cy, cz = BOX_CENTER
sx, sy, sz = BOX_SIZE

print('Binding site search box:')
print(f'  Center: X={cx}  Y={cy}  Z={cz}  (angstroms)')
print(f'  Size:   X={sx}  Y={sy}  Z={sz}  (angstroms)')
print(f'  Volume: {sx * sy * sz:.0f} cubic angstroms')
print(f'\nCoordinates derived from co-crystallized erlotinib position in {PDB_ID}.')
print('This targets the ATP binding pocket of the EGFR kinase domain.')

# Visualize the protein with py3Dmol
with open(str(pdb_path)) as f:
    pdb_data = f.read()

view = py3Dmol.view(width=750, height=450)
view.addModel(pdb_data, 'pdb')
view.setStyle({}, {'cartoon': {'color': 'lightgray', 'opacity': 0.8}})
view.addStyle({'resn': 'ERL'}, {'stick': {'colorscheme': 'orangeCarbon', 'radius': 0.25}})
# Highlight the co-crystallized ligand (erlotinib, residue name ERL)
view.addStyle({'resn': 'ERL'}, {'stick': {'colorscheme': 'orangeCarbon', 'radius': 0.3}})
view.zoomTo({'resn': 'ERL'})
view.setBackgroundColor('white')
view.show()
print('\nProtein structure shown above. Orange sticks = co-crystallized erlotinib in the binding pocket.')

## Cell 7: Run Docking for All Three Compounds

We now run AutoDock Vina for each compound. For each one, Vina explores thousands of different positions and orientations within the search box and returns the top binding poses ranked by estimated binding affinity in kcal/mol.

Binding affinity in docking is reported as a negative number. More negative means stronger predicted binding. As a rough reference, values more negative than -7 kcal/mol are generally considered promising for a drug-like molecule. Values around -5 or -6 are weak but detectable. Values less negative than -5 suggest the molecule does not fit the binding site well.

Each compound will take approximately 60 seconds to dock. The cell will print progress as each one finishes.

In [ ]:
# Cell 7 -- run AutoDock Vina for all three compounds

def run_docking(name, ligand_path):
    """Run Vina docking for one compound. Returns dict with scores and output path."""
    output_path = Path(f'{name}_docked.pdbqt')

    start = time.time()
    result = subprocess.run([
        'vina',
        '--receptor',      str(protein_pdbqt),
        '--ligand',        str(ligand_path),
        '--center_x',      str(BOX_CENTER[0]),
        '--center_y',      str(BOX_CENTER[1]),
        '--center_z',      str(BOX_CENTER[2]),
        '--size_x',        str(BOX_SIZE[0]),
        '--size_y',        str(BOX_SIZE[1]),
        '--size_z',        str(BOX_SIZE[2]),
        '--out',           str(output_path),
        '--exhaustiveness', str(EXHAUSTIVENESS),
        '--num_modes',      str(NUM_MODES),
    ], capture_output=True, text=True)
    elapsed = time.time() - start

    if result.returncode != 0:
        print(f'  {name}: Vina error')
        print(result.stderr)
        return None

    # Parse affinity scores from Vina output
    scores = []
    for line in result.stdout.split('\n'):
        parts = line.strip().split()
        if parts and parts[0].isdigit():
            try:
                scores.append(float(parts[1]))
            except (IndexError, ValueError):
                pass

    return {
        'name':         name,
        'best_score':   scores[0] if scores else None,
        'all_scores':   scores,
        'output_path':  output_path,
        'runtime_s':    elapsed,
        'stdout':       result.stdout,
    }


print(f'Running docking with exhaustiveness={EXHAUSTIVENESS}...')
print(f'Expected runtime: ~60 seconds per compound\n')

docking_results = {}

for c in compounds:
    name = c['name']
    if name not in ligand_paths:
        print(f'  {name}: skipped (ligand preparation failed)')
        continue
    print(f'  Docking {name}...', end=' ', flush=True)
    res = run_docking(name, ligand_paths[name])
    if res:
        docking_results[name] = res
        print(f'done. Best score: {res["best_score"]:.3f} kcal/mol  ({res["runtime_s"]:.0f}s)')

print(f'\nDocking complete. {len(docking_results)}/{len(compounds)} compounds docked successfully.')

## Cell 8: Compare Docking Scores

We now build a summary table comparing docking scores alongside the ADMET scores from the previous videos. This is the key comparison the notebook is designed to produce.

A good docking score confirms that a molecule not only has favorable drug-like properties but also fits into the target binding pocket. A compound with a high ADMET score but a weak docking score may be drug-like in general but not specifically active against EGFR. A compound with both a good ADMET score and a strong docking score is the most promising candidate to advance.

In [ ]:
# Cell 8 -- build comparison table and visualize docking vs ADMET scores

rows = []
for c in compounds:
    name = c['name']
    dr   = docking_results.get(name)
    rows.append({
        'Compound':       name,
        'Source':         c['source'],
        'ADMET Score':    c['admet_score'],
        'Docking (kcal/mol)': dr['best_score'] if dr else None,
        'Runtime (s)':    f"{dr['runtime_s']:.0f}" if dr else 'N/A',
    })

df_results = pd.DataFrame(rows)
df_results.to_csv(RESULTS_DIR / 'docking_results.csv', index=False)

print('Docking vs ADMET comparison:\n')
print(df_results[['Compound', 'ADMET Score', 'Docking (kcal/mol)', 'Source']].to_string(index=False))

# Bar chart comparing docking scores
names  = [r['Compound'] for r in rows if r['Docking (kcal/mol)'] is not None]
scores = [r['Docking (kcal/mol)'] for r in rows if r['Docking (kcal/mol)'] is not None]
colors = [c['color'] for c in compounds if c['name'] in names]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Docking scores
axes[0].barh(names, scores, color=colors, edgecolor='white')
axes[0].set_xlabel('Docking score (kcal/mol)', fontsize=11)
axes[0].set_title('Predicted Binding Affinity\n(more negative = stronger binding)', fontsize=11, fontweight='bold')
axes[0].axvline(x=-7.0, color='red', linestyle='--', linewidth=1, alpha=0.7, label='-7 kcal/mol threshold')
axes[0].legend(fontsize=9)
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)

# ADMET scores
admet_scores = [c['admet_score'] for c in compounds if c['name'] in names]
axes[1].barh(names, admet_scores, color=colors, edgecolor='white')
axes[1].set_xlabel('ADMET score (out of 10)', fontsize=11)
axes[1].set_title('ADMET Score from Video 7b/8\n(higher = better drug-like properties)', fontsize=11, fontweight='bold')
axes[1].set_xlim(0, 10)
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)

plt.suptitle('EGFR Candidate Compounds: Docking vs ADMET', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'docking_vs_admet.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Chart saved to {RESULTS_DIR}/docking_vs_admet.png')

## Cell 9: Visualize the Best Docked Pose

We now visualize the best docked pose for each compound using py3Dmol. The docked pose shows the predicted position and orientation of the ligand inside the EGFR binding pocket.

A good docked pose should sit deep inside the binding pocket, making contacts with the key residues that are known to be important for EGFR inhibitor binding. For EGFR, these include the hinge region residues Met793 and Gln791, the gatekeeper residue Thr790, and the hydrophobic pocket at the back of the ATP binding site.

We also overlay the co-crystallized erlotinib from the crystal structure as a reference. A compound that docks in a similar position and orientation to erlotinib is more likely to be a true EGFR inhibitor.

In [ ]:
# Cell 9 -- fixed visualization with better zoom and surface display

def visualize_docked_pose(compound_name, docked_pdbqt_path, pdb_path, title=''):
    with open(str(pdb_path)) as f:
        protein_data = f.read()
    with open(str(docked_pdbqt_path)) as f:
        ligand_data = f.read()

    # Extract only the first model (best pose)
    first_model = []
    in_model = False
    for line in ligand_data.split('\n'):
        if line.startswith('MODEL'):
            in_model = True
        if in_model:
            first_model.append(line)
        if line.startswith('ENDMDL') and in_model:
            break
    ligand_model1 = '\n'.join(first_model)

    view = py3Dmol.view(width=750, height=500)

    # Protein as semi-transparent surface so ligand is clearly visible
    view.addModel(protein_data, 'pdb')
    view.setStyle({'model': 0}, {'cartoon': {'color': 'lightgray', 'opacity': 0.5}})

    # Add surface around binding site only -- much easier to see ligand
    view.addSurface(py3Dmol.VDW,
                    {'opacity': 0.6, 'color': 'white'},
                    {'model': 0, 'resi': list(range(695, 760))})  # binding site residues in 1IEP

    # Co-crystallized erlotinib as orange reference
    view.addStyle({'model': 0, 'resn': 'ERL'},
                  {'stick': {'colorscheme': 'orangeCarbon', 'radius': 0.2}})

    # Docked compound as bold cyan sticks
    view.addModel(ligand_model1, 'pdbqt')
    view.setStyle({'model': 1}, {'stick': {'colorscheme': 'cyanCarbon', 'radius': 0.3}})

    # Zoom to ligand with generous padding so protein context is visible
    view.zoomTo({'model': 1})
    view.zoom(0.5)
    view.setBackgroundColor('white')

    print(title)
    print('Cyan sticks = docked compound  |  Orange sticks = co-crystallized erlotinib (reference)')
    view.show()


# Cell 9 -- simplest possible approach: extract ligand coords and build minimal PDB

for c in compounds:
    name = c['name']
    dr   = docking_results.get(name)
    if dr is None:
        continue

    with open(str(pdb_path)) as f:
        protein_data = f.read()
    with open(str(dr['output_path'])) as f:
        ligand_data = f.read()

    # Extract first model only
    first_model_lines = []
    in_model = False
    for line in ligand_data.split('\n'):
        if line.startswith('MODEL'):
            in_model = True
            continue
        if line.startswith('ENDMDL'):
            break
        if in_model and (line.startswith('ATOM') or line.startswith('HETATM') or line.startswith('BRANCH') or line.startswith('ROOT') or line.startswith('TORSDOF')):
            first_model_lines.append(line)

    # Convert PDBQT atom lines to plain PDB format for py3Dmol
    pdb_ligand_lines = []
    for line in first_model_lines:
        if line.startswith('ATOM') or line.startswith('HETATM'):
            # Rewrite as HETATM in plain PDB format
            pdb_ligand_lines.append('HETATM' + line[6:66])
    pdb_ligand_lines.append('END')
    ligand_pdb = '\n'.join(pdb_ligand_lines)

    view = py3Dmol.view(width=750, height=500)

    # Add protein
    view.addModel(protein_data, 'pdb')
    view.setStyle({}, {'cartoon': {'color': 'lightgray', 'opacity': 0.8}})
    view.addStyle({'resn': 'ERL'}, {'stick': {'colorscheme': 'orangeCarbon', 'radius': 0.25}})

    # Add ligand as plain PDB -- py3Dmol handles this much more reliably
    view.addModel(ligand_pdb, 'pdb')
    view.setStyle({'model': 1}, {'stick': {'colorscheme': 'cyanCarbon', 'radius': 0.35}})
    view.addStyle({'model': 1}, {'sphere': {'colorscheme': 'cyanCarbon', 'radius': 0.4}})

    # Zoom to ligand model only
    view.zoomTo({'model': 1})

    view.setBackgroundColor('white')

    score = dr['best_score']
    print(f"{name}  |  Score: {score:.3f} kcal/mol  |  ADMET: {c['admet_score']}/10")
    print('Cyan = docked compound  |  Orange = co-crystallized erlotinib (reference)')
    view.show()
    print()

## Cell 10: All Binding Poses per Compound

Vina returns multiple binding poses per compound, not just the best one. The top pose has the lowest energy, but looking at the spread of scores across all poses tells us something about how well the molecule fits the binding site.

A compound with a strong best score and a tight cluster of similar scores for the other poses is consistently finding the same binding mode, which gives us more confidence in the result. A compound where the scores spread widely across poses may be sampling multiple different orientations with similar energies, suggesting the binding mode is less well-defined.

In [ ]:
# Cell 10 -- display all binding poses and scores for each compound

print('All binding poses per compound:\n')

for c in compounds:
    name = c['name']
    dr   = docking_results.get(name)
    if dr is None:
        continue
    print(f"{name}  (ADMET score: {c['admet_score']}/10)")
    print(f"  {'Mode':<6} {'Affinity (kcal/mol)':<22} {'vs Best'}")
    print(f"  {'-'*45}")
    best = dr['all_scores'][0] if dr['all_scores'] else 0
    for i, score in enumerate(dr['all_scores'], 1):
        diff = score - best
        diff_str = f'+{diff:.3f}' if diff >= 0 else f'{diff:.3f}'
        print(f"  {i:<6} {score:<22.3f} {diff_str}")
    print()

# Save full results table
all_pose_rows = []
for c in compounds:
    dr = docking_results.get(c['name'])
    if dr:
        for i, score in enumerate(dr['all_scores'], 1):
            all_pose_rows.append({
                'compound':    c['name'],
                'admet_score': c['admet_score'],
                'mode':        i,
                'affinity':    score,
            })

df_all_poses = pd.DataFrame(all_pose_rows)
df_all_poses.to_csv(RESULTS_DIR / 'all_poses.csv', index=False)
print(f'All poses saved to {RESULTS_DIR}/all_poses.csv')

## Cell 11: Summary and Interpretation

We close the notebook with a summary that connects the docking results back to the full series narrative.

The key question this notebook was designed to answer is: do the improvements in ADMET score from Videos 7 and 8 translate into better binding at the EGFR binding site? The docking scores give us a partial answer. A compound that scores well on both ADMET and docking is the most credible candidate to take forward. A compound that scores well on ADMET but poorly on docking may be drug-like in general but not specifically targeting EGFR.

It is important to be honest about the limitations of docking at this stage. AutoDock Vina uses an empirical scoring function that approximates the free energy of binding. It does not account for protein flexibility, solvation effects, or entropic contributions in full detail. Docking scores are best used for relative ranking within a series of similar compounds, not as absolute predictions of binding affinity. The next step after a promising docking result in a real drug discovery program would be an experimental binding assay to confirm the computational prediction.

In [ ]:
# Cell 11 -- summary table and save all outputs

print('=' * 65)
print('SUMMARY: EGFR CANDIDATE DOCKING RESULTS')
print('=' * 65)
print(f'Protein structure: {PDB_ID} (EGFR kinase domain, co-crystallized with erlotinib)')
print(f'Docking engine:    AutoDock Vina v1.2.5')
print(f'Exhaustiveness:    {EXHAUSTIVENESS} (tutorial setting; use 16+ for publication)')
print(f'Search box:        {BOX_SIZE[0]} x {BOX_SIZE[1]} x {BOX_SIZE[2]} angstroms')
print(f'Center:            X={BOX_CENTER[0]} Y={BOX_CENTER[1]} Z={BOX_CENTER[2]}')
print()
print(f"{'Compound':<25} {'ADMET':>7} {'Docking':>14} {'Interpretation'}")
print('-' * 75)

for c in compounds:
    dr    = docking_results.get(c['name'])
    score = dr['best_score'] if dr else None
    if score is not None:
        if score < -7.5:
            interp = 'Strong predicted binding'
        elif score < -6.5:
            interp = 'Moderate predicted binding'
        else:
            interp = 'Weak predicted binding'
    else:
        interp = 'Docking failed'
        score  = float('nan')
    print(f"{c['name']:<25} {c['admet_score']:>5}/10 {score:>12.3f}  {interp}")

print()
print('Outputs saved to results/:')
for f in sorted(RESULTS_DIR.iterdir()):
    print(f'  {f.name}')

## Summary

In this notebook we ran a complete molecular docking pipeline for three EGFR inhibitor candidates carried forward from the series.

We fetched the EGFR crystal structure from the Protein Data Bank, prepared the protein using OpenBabel, generated 3D ligand conformers using RDKit, prepared ligand PDBQT files using Meeko, ran AutoDock Vina for all three compounds, and compared docking scores against the ADMET scores from the previous videos.

The comparison between ADMET scores and docking scores is the central output of this video. It illustrates a fundamental principle in computational drug discovery: no single model tells the whole story. ADMET models predict drug-like properties. Docking predicts target binding. A compound needs to score well on both to be a credible candidate for experimental validation.

**In the next video** we will bring together all the outputs from across this series into a structured end-to-end report: from target identification through virtual screening, ADMET prediction, molecular generation, literature mining, and molecular docking.